# Preprocesado y análisis de estacionariedad

Este notebook analiza la estacionariedad de las series utilizadas en el trabajo. Para cada ETF se estudian distintas variables, como el precio ajustado, el logaritmo del precio, el retorno simple, el retorno logarítmico, el volumen y el volumen en dólares.

El objetivo es comprobar qué representación de los datos es más adecuada para aplicar modelos de series temporales. A partir de los resultados de los tests ADF y KPSS, se identifican los ETFs cuyos retornos no presentan un comportamiento estacionario y se genera la lista final de activos válidos para los siguientes experimentos.

In [1]:
import pandas as pd

# Cargar precios ajustados
adj_close = pd.read_csv(
    "../Datos_csv/adj_close.csv",
    index_col=0,
    parse_dates=True
)

# Cargar volumen
volume = pd.read_csv(
    "../Datos_csv/volume.csv",
    index_col=0,
    parse_dates=True
)


La metodología se ilustra con un activo representativo. Seleccionamos la varibel principal (retornos en formato logarítmico)

In [2]:
import numpy as np
# Retornos
ret_simple = adj_close.pct_change()
ret_log = np.log(adj_close)
ret_log_diff = np.log(adj_close).diff() #diferencia con el dia anterior
dollar_vol = adj_close * volume


In [ ]:
#  Lista de ETFs 

tickers = list(adj_close.columns)
print("Número de ETFs:", len(tickers))
print(" Listado:", tickers)

Número de ETFs: 63
 Listado: ['AGG', 'BIL', 'BND', 'DBC', 'DIA', 'DVY', 'EEM', 'EFA', 'EWG', 'EWJ', 'EWQ', 'EWT', 'EWU', 'EWY', 'EWZ', 'FXI', 'GLD', 'HYG', 'IAU', 'IEF', 'IEFA', 'IEMG', 'IJR', 'INDA', 'ITOT', 'IVV', 'IWD', 'IWF', 'IWM', 'JNK', 'LQD', 'MCHI', 'MDY', 'MTUM', 'QQQ', 'QUAL', 'SCHD', 'SHY', 'SLV', 'SPY', 'TIP', 'TLT', 'USMV', 'USO', 'VEA', 'VIG', 'VLUE', 'VNQ', 'VOO', 'VTI', 'VTV', 'VUG', 'VWO', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']


Se analiza la estacionariedad de distintas transformaciones de la serie con el objetivo de identificar la representación más adecuada para la modelización temporal.

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning 
import warnings
warnings.filterwarnings("ignore", category=InterpolationWarning)

def tests_estacionariedad(serie):
    x = np.asarray(serie, float)
    x = x[~np.isnan(x)]
    #  ADF (H0: no estacionaria) 
    adf_stat, adf_p, adf_lags, adf_n, adf_crit, adf_icbest = adfuller(x, autolag='AIC')
    

    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(x, regression='c', nlags='auto' )
    if kpss_p <= 0.01 and adf_p>0.01: # rechazamos KPSS, pero no ADF
        s = "no estacionario"
    elif kpss_p > 0.01 and adf_p<=0.01: # rechazamos ADF, pero no KPSS 
        s = "estacionario"
    elif kpss_p > 0.01 and adf_p > 0.01: # no se rechaza nada        
        s = "no concluyente"
    else:
        s = "resultados inconsistentes" 
    return s

In [ ]:
# Estacionariedad (una fila por ETF y por serie)
estacionariedad_rows = []

for etf in tickers:

    #  Extraer series del ETF
  
    serie_adj_close     = adj_close[etf].dropna()
    serie_ret_simple    = ret_simple[etf].dropna()
    serie_log_precio    = ret_log[etf].dropna()
    serie_ret_log_diff  = ret_log_diff[etf].dropna()
    serie_dollar_vol    = dollar_vol[etf].dropna()
    serie_volume        = volume[etf].dropna()

    #  Estacionariedad

    series_a_testear = {
        "Adj_close": serie_adj_close,
        "log(Adj_close)": serie_log_precio,
        "retorno_simple": serie_ret_simple,
        "retorno_logaritmico": serie_ret_log_diff,
        "dollar_volume": serie_dollar_vol,
        "volume": serie_volume
    }

    res_est = []
    for nombre, s in series_a_testear.items():
        if len(s) < 50:
            estado = "insuficientes_datos"
        else:
            estado = tests_estacionariedad(s)

        row = {
            "ETF": etf,
            "Serie": nombre,
            "N_obs": len(s),
            "Inicio": s.index.min().date(),
            "Fin": s.index.max().date(),
            "Conclusión": estado
        }
        estacionariedad_rows.append(row)
        res_est.append(row)

    display(pd.DataFrame(res_est).head())

,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,AGG,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,AGG,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,AGG,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,AGG,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,AGG,dollar_volume,1276,2021-01-04,2026-02-02,resultados inconsistentes


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,BIL,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,BIL,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,BIL,retorno_simple,1275,2021-01-05,2026-02-02,no estacionario
3,BIL,retorno_logaritmico,1275,2021-01-05,2026-02-02,no estacionario
4,BIL,dollar_volume,1276,2021-01-04,2026-02-02,no estacionario


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,BND,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,BND,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,BND,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,BND,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,BND,dollar_volume,1276,2021-01-04,2026-02-02,resultados inconsistentes


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,DBC,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,DBC,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,DBC,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,DBC,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,DBC,dollar_volume,1276,2021-01-04,2026-02-02,no estacionario


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,DIA,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,DIA,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,DIA,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,DIA,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,DIA,dollar_volume,1276,2021-01-04,2026-02-02,resultados inconsistentes


...

,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,XLP,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,XLP,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,XLP,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,XLP,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,XLP,dollar_volume,1276,2021-01-04,2026-02-02,resultados inconsistentes


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,XLRE,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,XLRE,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,XLRE,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,XLRE,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,XLRE,dollar_volume,1276,2021-01-04,2026-02-02,estacionario


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,XLU,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,XLU,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,XLU,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,XLU,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,XLU,dollar_volume,1276,2021-01-04,2026-02-02,estacionario


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,XLV,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,XLV,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,XLV,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,XLV,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,XLV,dollar_volume,1276,2021-01-04,2026-02-02,resultados inconsistentes


,ETF,Serie,N_obs,Inicio,Fin,Conclusión
0,XLY,Adj_close,1276,2021-01-04,2026-02-02,no estacionario
1,XLY,log(Adj_close),1276,2021-01-04,2026-02-02,no estacionario
2,XLY,retorno_simple,1275,2021-01-05,2026-02-02,estacionario
3,XLY,retorno_logaritmico,1275,2021-01-05,2026-02-02,estacionario
4,XLY,dollar_volume,1276,2021-01-04,2026-02-02,no concluyente


ELIMINAMOS LOS ETFS NO ESTACIONARIOS EN RETORNOS

In [ ]:

# Convertimos la lista acumulada en DataFrame
estacionariedad_df = pd.DataFrame(estacionariedad_rows)

# ETFs válidos
etfs_validos = []

# ETFs eliminados
etfs_eliminados = []

for etf in tickers:
    
    # Filtramos solo las filas de ese ETF
    df_etf = estacionariedad_df[estacionariedad_df["ETF"] == etf]
    
    ret_simple_estado = df_etf[df_etf["Serie"] == "retorno_simple"]["Conclusión"].values
    ret_log_estado    = df_etf[df_etf["Serie"] == "retorno_logaritmico"]["Conclusión"].values
    
    if (
        len(ret_simple_estado) > 0 and
        len(ret_log_estado) > 0 and
        ret_simple_estado[0] == "estacionario" or
        ret_log_estado[0] == "estacionario"
    ):
        etfs_validos.append(etf)
    else:
        etfs_eliminados.append(etf)

print("Número total ETFs originales:", len(tickers))
print("Número ETFs válidos:", len(etfs_validos))
print("Número ETFs eliminados:", len(etfs_eliminados))

print("\nETFs eliminados:")
display(pd.DataFrame(etfs_eliminados, columns=["ETF eliminado"]))

tickers_filtrados = etfs_validos


Número total ETFs originales: 63
Número ETFs válidos: 58
Número ETFs eliminados: 5

ETFs eliminados:


,ETF eliminado
0,BIL
1,EWY
2,GLD
3,IAU
4,SHY


In [7]:
import json
import os

resultado = {
    "tickers_filtrados": tickers_filtrados,
    "etfs_eliminados": etfs_eliminados
}

out_path = "../Datos_csv/tickers_estacionalidad.json"
with open(out_path, 'w', encoding='utf-8') as fout:
    json.dump(resultado, fout, indent=2, ensure_ascii=False)

print(f"Guardado en: {out_path}")
print(f"ETFs validos  : {len(tickers_filtrados)}")
print(f"ETFs eliminados: {etfs_eliminados}")

Guardado en: ../Datos_csv/tickers_estacionalidad.json
ETFs validos  : 58
ETFs eliminados: ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']
